<a href="https://colab.research.google.com/github/CelineShwkan/cloudProject/blob/main/Cloud_Tutorial_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [1]:

# Google Colab - Tutorial 1
# Student Information Form

import os
import csv
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive


# Connect to Google Drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")


# Find the existing students file
DRIVE_FOLDER = Path("/content/drive/MyDrive")

# Set a full path here if needed
STUDENTS_FILE = DRIVE_FOLDER / "students.txt"

if not STUDENTS_FILE.is_file():
    matches = list(DRIVE_FOLDER.rglob("students.txt"))

    if len(matches) == 1:
        STUDENTS_FILE = matches[0]
    elif len(matches) == 0:
        raise FileNotFoundError(
            "students.txt was not found in Google Drive."
        )
    else:
        print("Multiple student files were found:")
        for path in matches:
            print(path)
        raise ValueError(
            "Set STUDENTS_FILE to the correct full path."
        )

print("Connected file:", STUDENTS_FILE)


# Detect the separator in the student file
def detect_separator():
    lines = STUDENTS_FILE.read_text(
        encoding="utf-8-sig"
    ).splitlines()

    first_line = next(
        (line for line in lines if line.strip()),
        None
    )

    if first_line is None:
        raise ValueError("The student file is empty.")

    for sep in ("|", ";", "\t", ","):
        row = next(csv.reader(
            [first_line],
            delimiter=sep,
            skipinitialspace=True
        ))

        if len(row) in (5, 6):
            return sep

    raise ValueError(
        "Invalid file format. Each student must have "
        "5 or 6 separated fields."
    )


separator = detect_separator()


# Read students from the existing file
def load_students():
    students = {}

    with STUDENTS_FILE.open(
        "r", encoding="utf-8-sig", newline=""
    ) as file:

        reader = csv.reader(
            file,
            delimiter=separator,
            skipinitialspace=True
        )

        for line_number, row in enumerate(reader, 1):

            if not row or not any(row):
                continue

            if len(row) not in (5, 6):
                raise ValueError(
                    f"Invalid data on line {line_number}"
                )

            first, last, email, courses, link = [
                value.strip() for value in row[:5]
            ]

            favorite = (
                row[5].strip() if len(row) == 6 else ""
            )

            if not all([first, last, email, courses, link]):
                raise ValueError(
                    f"Missing data on line {line_number}"
                )

            # Keep the latest record for each email
            students[email] = {
                "first": first,
                "last": last,
                "email": email,
                "courses": courses,
                "link": link,
                "favorite": favorite
            }

    if not students:
        raise ValueError("No students were found.")

    return students


students = load_students()


# Form layout and style
field_style = {"description_width": "130px"}
field_layout = widgets.Layout(width="650px")


# Student selection dropdown
student_dropdown = widgets.Dropdown(
    options=[
        (
            s["first"] + " " + s["last"],
            email
        )
        for email, s in students.items()
    ],
    description="סטודנט:",
    style=field_style,
    layout=field_layout
)


# Student information text fields
first_name = widgets.Text(
    description="שם פרטי:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

last_name = widgets.Text(
    description="שם משפחה:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

student_email = widgets.Text(
    description="מייל:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

student_courses = widgets.Textarea(
    description="קורסים:",
    disabled=True,
    style=field_style,
    layout=widgets.Layout(
        width="650px",
        height="75px"
    )
)

student_link = widgets.Text(
    description="קישור מעניין:",
    disabled=True,
    style=field_style,
    layout=field_layout
)


# Favorite TV show input
favorite_show = widgets.Text(
    description="תוכנית אהובה:",
    placeholder="Enter favorite TV show",
    style=field_style,
    layout=field_layout
)


# Update button
update_button = widgets.Button(
    description="עדכון התוכנית",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="200px")
)


# Output message
output = widgets.Output()


# Display selected student information
def on_student_change(change=None):

    student = students[student_dropdown.value]

    first_name.value = student["first"]
    last_name.value = student["last"]
    student_email.value = student["email"]
    student_courses.value = student["courses"]
    student_link.value = student["link"]
    favorite_show.value = student["favorite"]

    with output:
        clear_output()


# Update favorite show in Google Drive
def on_update_clicked(button):

    favorite = favorite_show.value.strip()

    with output:
        clear_output()

        if not favorite:
            print("Please enter a favorite TV show.")
            return

        if "\n" in favorite or "\r" in favorite:
            print("Invalid TV show name.")
            return

        student = students[student_dropdown.value]

        # Create a new record with the updated show
        new_record = [
            student["first"],
            student["last"],
            student["email"],
            student["courses"],
            student["link"],
            favorite
        ]

        try:
            # Check whether the file ends with a newline
            with STUDENTS_FILE.open("rb") as file:
                file.seek(0, 2)
                needs_newline = False

                if file.tell() > 0:
                    file.seek(-1, 2)
                    needs_newline = file.read(1) != b"\n"

            # Append a new line without deleting old data
            with STUDENTS_FILE.open(
                "a",
                encoding="utf-8",
                newline=""
            ) as file:

                if needs_newline:
                    file.write("\n")

                writer = csv.writer(
                    file,
                    delimiter=separator,
                    lineterminator="\n"
                )

                writer.writerow(new_record)

            # Update the current form data
            student["favorite"] = favorite

            print("Student updated successfully!")
            print("Favorite show:", favorite)
            print("Saved to Google Drive.")

        except OSError as error:
            print("Could not save the update:", error)


# Connect widget events
student_dropdown.observe(
    on_student_change,
    names="value"
)

update_button.on_click(on_update_clicked)


# Display the initial student
on_student_change()


# Create the complete form
form_title = widgets.HTML(
    value="""
    <h2 style="color:#243b53;">
        Student Information Form
    </h2>
    <p>Select a student and update their favorite TV show.</p>
    """
)


form = widgets.VBox(
    [
        form_title,
        student_dropdown,
        first_name,
        last_name,
        student_email,
        student_courses,
        student_link,
        favorite_show,
        update_button,
        output
    ],
    layout=widgets.Layout(
        border="1px solid #dddddd",
        padding="25px",
        width="720px"
    )
)


# Show the form
display(form)


Mounted at /content/drive
Connected file: /content/drive/MyDrive/students.txt


ValueError: Invalid file format. Each student must have 5 or 6 separated fields.

In [2]:

# Google Colab - Tutorial 1
# Student Information Form

import os
import csv
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive


# Connect to Google Drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")


# Find the existing students file
DRIVE_FOLDER = Path("/content/drive/MyDrive")

# Set a full path here if needed
STUDENTS_FILE = DRIVE_FOLDER / "students.txt"

if not STUDENTS_FILE.is_file():
    matches = list(DRIVE_FOLDER.rglob("students.txt"))

    if len(matches) == 1:
        STUDENTS_FILE = matches[0]
    elif len(matches) == 0:
        raise FileNotFoundError(
            "students.txt was not found in Google Drive."
        )
    else:
        print("Multiple student files were found:")
        for path in matches:
            print(path)
        raise ValueError(
            "Set STUDENTS_FILE to the correct full path."
        )

print("Connected file:", STUDENTS_FILE)


# Detect the separator in the student file
def detect_separator():
    lines = STUDENTS_FILE.read_text(
        encoding="utf-8-sig"
    ).splitlines()

    first_line = next(
        (line for line in lines if line.strip()),
        None
    )

    if first_line is None:
        raise ValueError("The student file is empty.")

    for sep in ("|", ";", "\t", ","):
        row = next(csv.reader(
            [first_line],
            delimiter=sep,
            skipinitialspace=True
        ))

        if len(row) in (5, 6):
            return sep

    raise ValueError(
        "Invalid file format. Each student must have "
        "5 or 6 separated fields."
    )


separator = detect_separator()


# Read students from the existing file
def load_students():
    students = {}

    with STUDENTS_FILE.open(
        "r", encoding="utf-8-sig", newline=""
    ) as file:

        reader = csv.reader(
            file,
            delimiter=separator,
            skipinitialspace=True
        )

        for line_number, row in enumerate(reader, 1):

            if not row or not any(row):
                continue

            if len(row) not in (5, 6):
                raise ValueError(
                    f"Invalid data on line {line_number}"
                )

            first, last, email, courses, link = [
                value.strip() for value in row[:5]
            ]

            favorite = (
                row[5].strip() if len(row) == 6 else ""
            )

            if not all([first, last, email, courses, link]):
                raise ValueError(
                    f"Missing data on line {line_number}"
                )

            # Keep the latest record for each email
            students[email] = {
                "first": first,
                "last": last,
                "email": email,
                "courses": courses,
                "link": link,
                "favorite": favorite
            }

    if not students:
        raise ValueError("No students were found.")

    return students


students = load_students()


# Form layout and style
field_style = {"description_width": "130px"}
field_layout = widgets.Layout(width="650px")


# Student selection dropdown
student_dropdown = widgets.Dropdown(
    options=[
        (
            s["first"] + " " + s["last"],
            email
        )
        for email, s in students.items()
    ],
    description="סטודנט:",
    style=field_style,
    layout=field_layout
)


# Student information text fields
first_name = widgets.Text(
    description="שם פרטי:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

last_name = widgets.Text(
    description="שם משפחה:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

student_email = widgets.Text(
    description="מייל:",
    disabled=True,
    style=field_style,
    layout=field_layout
)

student_courses = widgets.Textarea(
    description="קורסים:",
    disabled=True,
    style=field_style,
    layout=widgets.Layout(
        width="650px",
        height="75px"
    )
)

student_link = widgets.Text(
    description="קישור מעניין:",
    disabled=True,
    style=field_style,
    layout=field_layout
)


# Favorite TV show input
favorite_show = widgets.Text(
    description="תוכנית אהובה:",
    placeholder="Enter favorite TV show",
    style=field_style,
    layout=field_layout
)


# Update button
update_button = widgets.Button(
    description="עדכון התוכנית",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="200px")
)


# Output message
output = widgets.Output()


# Display selected student information
def on_student_change(change=None):

    student = students[student_dropdown.value]

    first_name.value = student["first"]
    last_name.value = student["last"]
    student_email.value = student["email"]
    student_courses.value = student["courses"]
    student_link.value = student["link"]
    favorite_show.value = student["favorite"]

    with output:
        clear_output()


# Update favorite show in Google Drive
def on_update_clicked(button):

    favorite = favorite_show.value.strip()

    with output:
        clear_output()

        if not favorite:
            print("Please enter a favorite TV show.")
            return

        if "\n" in favorite or "\r" in favorite:
            print("Invalid TV show name.")
            return

        student = students[student_dropdown.value]

        # Create a new record with the updated show
        new_record = [
            student["first"],
            student["last"],
            student["email"],
            student["courses"],
            student["link"],
            favorite
        ]

        try:
            # Check whether the file ends with a newline
            with STUDENTS_FILE.open("rb") as file:
                file.seek(0, 2)
                needs_newline = False

                if file.tell() > 0:
                    file.seek(-1, 2)
                    needs_newline = file.read(1) != b"\n"

            # Append a new line without deleting old data
            with STUDENTS_FILE.open(
                "a",
                encoding="utf-8",
                newline=""
            ) as file:

                if needs_newline:
                    file.write("\n")

                writer = csv.writer(
                    file,
                    delimiter=separator,
                    lineterminator="\n"
                )

                writer.writerow(new_record)

            # Update the current form data
            student["favorite"] = favorite

            print("Student updated successfully!")
            print("Favorite show:", favorite)
            print("Saved to Google Drive.")

        except OSError as error:
            print("Could not save the update:", error)


# Connect widget events
student_dropdown.observe(
    on_student_change,
    names="value"
)

update_button.on_click(on_update_clicked)


# Display the initial student
on_student_change()


# Create the complete form
form_title = widgets.HTML(
    value="""
    <h2 style="color:#243b53;">
        Student Information Form
    </h2>
    <p>Select a student and update their favorite TV show.</p>
    """
)


form = widgets.VBox(
    [
        form_title,
        student_dropdown,
        first_name,
        last_name,
        student_email,
        student_courses,
        student_link,
        favorite_show,
        update_button,
        output
    ],
    layout=widgets.Layout(
        border="1px solid #dddddd",
        padding="25px",
        width="720px"
    )
)


# Show the form
display(form)


Connected file: /content/drive/MyDrive/students.txt


In [ ]:

!pip install -q ipywidgets numpy matplotlib


In [ ]:
import time
print(time.ctime())


Sun Nov  1 09:23:55 2026


In [ ]:
!pip install numpy


In [ ]:

from pathlib import Path

FOLDER = Path("/content/drive/MyDrive/Cloud_Tutorial_1")
FOLDER.mkdir(parents=True, exist_ok=True)

STUDENTS_FILE = FOLDER / "students.txt"

print("Folder:", FOLDER)


Folder: /content/drive/MyDrive/Cloud_Tutorial_1


In [ ]:
!ls "/content/drive/My Drive/Colab Notebooks"


Cloud_Tutorial_1.ipynb


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

y = np.random.randn(100)
x = [x for x in range(len(y))]

plt.plot(x, y, '-')
plt.fill_between(x, y, 200, where = (y > 195), facecolor='g', alpha=0.6)

plt.title("Sample Plot")
plt.show()


name = 'Cloud course’
#@param {type:"string"} print(name)


#@title Date fields
date_input = '2019-06-03' #@param {type:"date"}
print(date_input)


SyntaxError: unterminated string literal (detected at line 14) (1711640513.py, line 14)

In [ ]:

from pathlib import Path

STUDENTS_FILE = Path("/content/students.txt")

print("File exists:", STUDENTS_FILE.exists())


File exists: True


In [ ]:

with open(STUDENTS_FILE, "r", encoding="utf-8") as file:
    contents = file.read()
    print(contents)


(div3)
pow3=1
i=0
(FOR)
	if (i==10) goto ENDFOR
	*(sp+4)+i=pow3
	currentpow3 = pow3
	pow3 = pow3 + pow3
	pow3 = pow3 + cuurentpow3
	i++
	goto FOR
(ENDFOR)
n=19683
i=9
q1=0
(WHILE1)
	if i=<0 goto ENDWHILE1
	(WHILE2)
	   if x<n goto ENDWHILW2
	   x=x-n
	   q1=q1+*(sp+4)+(i-1)
	   goto WHILE2
	(ENDWHILW2)
	n=*(sp+4)+(i-1)
	i=i-1
goto WHILE1
(ENDWHILE1)
*(SP+3)=*(SP+1)





	
	
	
		   
   
   

